In [ ]:
# -------------------------------------------------------------------
# Запуск Demucs на одном файле для теста вне API
# -------------------------------------------------------------------

import os
import subprocess

# Укажите путь к Вашему аудиофайлу
INPUT_FILE = "your_audio_file.mp3" # замените на свой путь
OUTPUT_DIR = "output_folder"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Команда запуска Demucs с экспортом в MP3
cmd = [
    sys.executable, "-m", "demucs",
    "-n", "htdemucs",
    "-o", OUTPUT_DIR,
    "--mp3",
    "--mp3-bitrate", "192",
    INPUT_FILE
]

print("Running Demucs...")

# Выполнение команды Demucs с захватом вывода
result = subprocess.run(cmd, text=True, capture_output=True)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Demucs failed")

print("DONE")

In [ ]:
# -------------------------------------------------------------------
# Оценка качества разделения на датасете MUSDB18
# -------------------------------------------------------------------

import os
import sys
import subprocess
import numpy as np
import musdb
import soundfile as sf
import pandas as pd
from mir_eval.separation import bss_eval_sources

# Укажите путь к MUSDB18
MUSDB_PATH = "путь_к_папке_MUSDB18" # замените на свой путь
OUT_DIR = "demucs_output_eval"

os.makedirs(OUT_DIR, exist_ok=True)

N_TRACKS = 5
CUT_SECONDS = 30
SR = 44100

# Загрузка тестовых треков из MUSDB18
db = musdb.DB(root=MUSDB_PATH, subsets="test")
tracks = db.tracks[:N_TRACKS]

print("Tracks:", len(tracks))

targets = ["vocals", "drums", "bass", "other"]

# Преобразование стерео в моно
def mono(x):
    return np.mean(x, axis=1).astype(np.float32)

# Обрезка аудио до заданной длины
def cut(x):
    return x[:CUT_SECONDS * SR]

# Загрузка и преобразование в моно предсказанного стема
def load_est(path):
    x, _ = sf.read(path)
    return mono(x)

# Список для сбора метрик по трекам
rows = []

# Цикл оценки качества каждого трека
for track in tracks:
    print("\nProcessing:", track.name)

    mix = cut(track.audio)

    temp_path = os.path.join(OUT_DIR, "_temp.wav")
    sf.write(temp_path, mix, track.rate)

    # Запуск Demucs для разделения микса (вывод скрыт)
    subprocess.run([
        sys.executable, "-m", "demucs",
        "-n", "htdemucs",
        "-o", OUT_DIR,
        "--mp3",
        "--mp3-bitrate", "192",
        temp_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    folder = os.path.join(OUT_DIR, "htdemucs", "_temp")

    refs = []
    ests = []

    valid = True

    # Сравнение референсного и предсказанного стемов с обрезкой до одинаковой длины
    for t in targets:
        est_path = os.path.join(folder, f"{t}.mp3")

        if not os.path.exists(est_path):
            valid = False
            break

        ref = mono(cut(track.targets[t].audio))
        est = load_est(est_path)

        L = min(len(ref), len(est))
        if L < 1000:
            valid = False
            break

        refs.append(ref[:L])
        ests.append(est[:L])

    if not valid:
        continue

    refs = np.stack(refs)
    ests = np.stack(ests)

    sdr, sir, sar, _ = bss_eval_sources(refs, ests)

    # Сохранение метрик для каждого трека
    for i, t in enumerate(targets):
        rows.append({
            "track": track.name,
            "target": t,
            "SDR": float(sdr[i]),
            "SIR": float(sir[i]),
            "SAR": float(sar[i]),
        })

# Создание таблицы с результатами оценки
df = pd.DataFrame(rows)

print("\nRESULT:")
print(df.head())

print("\nMEAN METRICS:")
print(df.groupby("target")[["SDR", "SIR", "SAR"]].mean())

In [ ]:
# -------------------------------------------------------------------
# Визуализация средних метрик оценки качества
# -------------------------------------------------------------------

import matplotlib.pyplot as plt

# Вычисление средних значений метрик по каждому типу стема
mean = df.groupby("target")[["SDR", "SIR", "SAR"]].mean()

# Построение графика среднего SDR по стемам
plt.figure()
mean["SDR"].plot(kind="bar")
plt.title("Mean SDR by target")
plt.ylabel("SDR")
plt.show()

# Построение графика среднего SIR по стемам
plt.figure()
mean["SIR"].plot(kind="bar")
plt.title("Mean SIR by target")
plt.ylabel("SIR")
plt.show()

# Построение графика среднего SAR по стемам
plt.figure()
mean["SAR"].plot(kind="bar")
plt.title("Mean SAR by target")
plt.ylabel("SAR")
plt.show()